### Import necessary packages and modules

In [1]:
import os, random, numpy as np, torch
import matplotlib.pyplot as plt
import time
import gymnasium as gym
from gymnasium import spaces
from typing import Callable, Optional, Dict, Any, Tuple
import numpy as np

from oceanrl import query  # query(salmon, shark, effort, month)

# Stable-Baselines3
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement

SEED = 2025
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Define Fishing Environment

In [2]:
class FishingEnvContinuous(gym.Env):
    metadata = {"render_modes": []}

    def __init__(
        self,
        *,
        salmon_t0: float,
        shark_t0: float,
        query: Callable[[float, float, float, int], Tuple[float, float, float]],
        max_month: int = 900,
        # reward weights
        K1: float = 1e-3,
        K2: float = 1e-2,
        K3: float = 100.0,
        K4: float = 100.0,
        # obs normalization
        obs_mode: str = "ratio",
        ema_decay: float = 0.995,
        clip_factor: float = 5.0,
        # optional manual cap, when it is None, there is no cap as required in this project
        effort_cap: Optional[float] = None,
        effort_transform_eps: float = 1e-6,
        seed: Optional[int] = None,
    ):
        super().__init__()
        self.rng = np.random.default_rng(seed)

        self.salmon0 = float(salmon_t0)
        self.shark0 = float(shark_t0)
        self.max_month = int(max_month)
        self.query = query

        self.K1, self.K2, self.K3, self.K4 = float(K1), float(K2), float(K3), float(K4)

        # optional manual cap, when it is None, there is no cap as required in this project
        self.effort_cap = float(effort_cap) if effort_cap is not None else None
        self.effort_transform_eps = float(effort_transform_eps)
        self._tan_scale = np.pi / 2 - self.effort_transform_eps

        # normalisation helpers
        self.obs_mode = obs_mode
        self.ema_decay = float(ema_decay)
        self.clip_factor = float(clip_factor)
        self.s_ema = max(self.salmon0, 1e-6)
        self.k_ema = max(self.shark0, 1e-6)

        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(3,), dtype=np.float32)
        # policy outputs actions in [-1, 1] and then map to [0, ∞) via tan transform
        self.action_space = spaces.Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32,
        )

        self.salmon = self.salmon0
        self.shark = self.shark0
        self.month = 1

    #  define helpers 
    def _a_to_effort(self, a: np.ndarray) -> float:
        a_scalar = float(np.clip(a[0], -1.0 + self.effort_transform_eps, 1.0 - self.effort_transform_eps))
        u = 0.5 * (a_scalar + 1.0)  # map to (0, 1)
        effort = float(np.tan(self._tan_scale * u))
        if self.effort_cap is not None:
            effort = min(effort, self.effort_cap)
        return max(effort, 0.0)

    def _step_reward(self, caught: float, effort: float) -> float:
        return self.K1 * float(caught) - self.K2 * float(effort)

    def _terminal_bonus(self) -> float:
        eps = 1e-10
        return self.K3 * np.log(max(self.salmon, eps)) + self.K4 * np.log(max(self.shark, eps))

    def _update_scales(self, s: float, k: float):
        self.s_ema = self.ema_decay * self.s_ema + (1 - self.ema_decay) * max(s, 1e-6)
        self.k_ema = self.ema_decay * self.k_ema + (1 - self.ema_decay) * max(k, 1e-6)

    def _norm_obs(self, s: float, k: float, m: int) -> np.ndarray:
        self._update_scales(s, k)
        if self.obs_mode == "log":
            s_n = np.log1p(s) / np.log1p(self.clip_factor * self.s_ema)
            k_n = np.log1p(k) / np.log1p(self.clip_factor * self.k_ema)
        else:
            s_n = s / (self.clip_factor * self.s_ema)
            k_n = k / (self.clip_factor * self.k_ema)
        s_n = float(np.clip(s_n, 0.0, 1.0))
        k_n = float(np.clip(k_n, 0.0, 1.0))
        m_n = float(np.clip(m / self.max_month, 0.0, 1.0))
        return np.array([s_n, k_n, m_n], dtype=np.float32)

    #  gym API 
    def reset(self, *, seed: Optional[int] = None, options: Optional[Dict[str, Any]] = None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.salmon = self.salmon0
        self.shark = self.shark0
        self.month = 1
        self.s_ema = max(self.salmon0, 1e-6)
        self.k_ema = max(self.shark0, 1e-6)
        obs = self._norm_obs(self.salmon, self.shark, self.month)
        return obs, {}

    def step(self, action: np.ndarray):
        effort = self._a_to_effort(action)
        sc, s1, k1 = self.query(self.salmon, self.shark, float(effort), self.month)
        reward = self._step_reward(sc, effort)
        self.salmon, self.shark = float(s1), float(k1)
        self.month += 1
        terminated = self.month > self.max_month
        if terminated:
            reward += self._terminal_bonus()
        obs = self._norm_obs(self.salmon, self.shark, self.month)
        info = {"effort": float(effort), "salmon_caught": float(sc)}
        if self.effort_cap is not None:
            info["effort_cap"] = float(self.effort_cap)
        return obs, float(reward), bool(terminated), False, info

    def render(self):
        print(f"t={self.month:3d} | salmon={self.salmon:.3f} | shark={self.shark:.3f}")



## Warp the B3 SAC Model into SAC Agent

In [3]:
class SACAgent:
    def __init__(self, model_path="./best_sac/best_model.zip", env=None):
        """
        model_path: path to trained SAC model
        env: an instance of FishingEnvContinuous (needed to use the same normalization & effort transform)
        """
        self.model = SAC.load(model_path)
        self.env = env
        if env is None:
            raise ValueError("You must pass in the environment used for normalization & effort transform.")

    def act(self, state):
        """
        state: (salmon, shark, month) in raw unnormalized form.
        returns: actual fishing effort (float)
        """
        salmon, shark, month = state

        # --- Apply SAME normalization your SAC model was trained with ---
        obs = self.env._norm_obs(float(salmon), float(shark), int(month))
        obs = obs.reshape(1, -1)

        # --- SAC predicts action in [-1, 1] ---
        action, _ = self.model.predict(obs, deterministic=True)

        # --- Convert SAC action → actual fishing effort ---
        fishing_effort = self.env._a_to_effort(action)

        return float(fishing_effort)


In [4]:
env = FishingEnvContinuous(
    salmon_t0=20000.0,
    shark_t0=500.0,
    query=query,
    max_month=900,
    K1=1e-3, K2=1e-2, K3=100.0, K4=100.0,
    obs_mode="ratio",
    ema_decay=0.995,
    clip_factor=5.0,
    effort_cap=None,
    seed=SEED,
)

In [5]:
## During evaluation on week 10, will perform inference on different starting combinations
agent = SACAgent("./best_sac/best_model.zip", env=env)

## Initial values for 1st month
salmon_t = 20000
shark_t = 500
total_salmon_caught = 0
total_effort = 0

for month_t in range(1, 901):
    fishing_effort_t = agent.act((salmon_t, shark_t, month_t))
    salmon_caught_t, salmon_t_plus_1, shark_t_plus_1 = query(salmon_t, shark_t, fishing_effort_t, month_t)
    total_salmon_caught += salmon_caught_t
    total_effort += fishing_effort_t
    salmon_t, shark_t = salmon_t_plus_1, shark_t_plus_1

K1, K2, K3, K4 = 0.001, 0.01, 100, 100
G = K1*total_salmon_caught - K2*total_effort + K3*np.log(salmon_t+1e-10) + K4*np.log(shark_t+1e-10)

print(G)

C:\Users\pakke\anaconda3\envs\p312\Lib\site-packages\stable_baselines3\common\save_util.py:167: UserWarning: Could not deserialize object lr_schedule. Consider using `custom_objects` argument to replace this object.
Exception: Can't get attribute 'FloatSchedule' on <module 'stable_baselines3.common.utils' from 'C:\\Users\\pakke\\anaconda3\\envs\\p312\\Lib\\site-packages\\stable_baselines3\\common\\utils.py'>
  warnings.warn(
C:\Users\pakke\AppData\Local\Temp\ipykernel_57736\1141634832.py:61: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  a_scalar = float(np.clip(a[0], -1.0 + self.effort_transform_eps, 1.0 - self.effort_transform_eps))


10297.013985039459
